# AOI-PCB-SSD: Inference & Evaluation

Loads a trained model, evaluates it on the held-out validation split, and
visualises predicted vs. ground-truth IC corner points.

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Train a model: `python scripts/train.py --architecture custom`

### Contents
1. Setup
2. Load model
3. Validation data
4. Evaluate
5. Predict & decode
6. Visualise predictions

## 1. Setup

Set `MODEL_PATH` to a specific `model.keras`, or leave it as `None` to
auto-select the most recently modified run in `experiments/`. The matching
`config.json` saved alongside the model is loaded automatically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

from aoi_pcb_ssd.config_loader import Config
from aoi_pcb_ssd.data.data_generator import DataGenerator
from aoi_pcb_ssd.encoding.input_encoder import SSDInputEncoder
from aoi_pcb_ssd.encoding.output_decoder import decode_detections
from aoi_pcb_ssd.model.loss import AOILoss
from aoi_pcb_ssd.model.metrics import class_mAP, mae

MODEL_PATH = None  # set explicitly, or auto-select the most recent run

if MODEL_PATH is None:
    runs = sorted(
        Path("../experiments").glob("*/model.keras"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not runs:
        raise FileNotFoundError("No trained model in ../experiments/. Run scripts/train.py first.")
    model_path = runs[0]
else:
    model_path = Path(MODEL_PATH)

config = Config(str(model_path.parent / "config.json"))
m = config.model
t = config.training
print(f"Model: {model_path}")

## 2. Load Model

The custom loss is a bound method that Keras cannot deserialise from the saved
compile config, so the model is loaded with `compile=False` and re-compiled
immediately. `GridCenters` is decorated with `@register_keras_serializable` and
is resolved automatically from the registry — no `custom_objects` kwarg needed.


In [ ]:
aoi_loss = AOILoss(**config.get_init_kwargs("training.loss"))
model = tf.keras.models.load_model(
    str(model_path),
    compile=False,
)
model.compile(
    optimizer=Adam(**config.get_init_kwargs("training.optimizer")),
    loss=aoi_loss.compute_loss,
    metrics=[class_mAP, mae],
)
model.summary()

## 3. Validation Data

The same 80/20 shuffled split (seeded from the config) used during training is
reconstructed, so evaluation runs on the held-out validation images only.

In [ ]:
encoder = SSDInputEncoder(
    img_height=m.img_height,
    img_width=m.img_width,
    n_classes=m.n_classes,
    predictor_sizes=m.predictor_sizes,
    normalize_coords=m.normalize_coords,
)

generator = DataGenerator(
    parent_dir=str(Path("..") / config.generator.train_data.crop_save_dir),
    encoder=encoder,
    augmentation=False,
)
X, y = generator.get_data()

_, X_val, _, y_val = train_test_split(
    X, y, test_size=t.val_split, shuffle=True, random_state=t.random_seed
)
print(f"Validation: {X_val.shape}")

## 4. Evaluate

In [ ]:
results = model.evaluate(
    X_val, y_val, batch_size=t.batch_size, verbose=1, return_dict=True
)
for name, value in results.items():
    print(f"  {name}: {value:.6f}")

## 5. Predict & Decode

`decode_detections` reverses the encoding: it reads the predicted class and
confidence, adds each anchor centre back to its corner offsets, denormalises to
pixel coordinates, and drops background cells.

In [ ]:
predictions = model.predict(X_val, batch_size=t.batch_size, verbose=0)

decode_kwargs = dict(
    normalize_coords=m.normalize_coords,
    img_height=m.img_height,
    img_width=m.img_width,
)
gt_dets = decode_detections(y_val, **decode_kwargs)
pred_dets = decode_detections(predictions, **decode_kwargs)
print(f"Predictions: {predictions.shape}")

## 6. Visualise Predictions

Ground-truth corners are drawn in **red**, predicted corners in **blue**, with
the predicted class and confidence labelled at each IC centre.

In [ ]:
_CLASSES = ["background", "ic"]

n_show = min(8, len(X_val))
cols = 4
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

for idx, ax in enumerate(axes.flat):
    if idx >= n_show:
        ax.axis("off")
        continue
    ax.imshow(X_val[idx])
    for det in gt_dets[idx]:
        for x, y_ in det[2:].reshape(-1, 2):
            ax.add_patch(plt.Circle((x, y_), 3, color="red", fill=False, lw=1.5))
    for det in pred_dets[idx]:
        corners = det[2:].reshape(-1, 2)
        for x, y_ in corners:
            ax.add_patch(plt.Circle((x, y_), 3, color="deepskyblue", fill=False, lw=1.5))
        cx = (corners[0, 0] + corners[3, 0]) / 2
        cy = (corners[0, 1] + corners[3, 1]) / 2
        ax.text(
            cx, cy, f"{_CLASSES[int(det[0])]}: {det[1]:.2f}",
            color="white", fontsize=7,
            bbox=dict(facecolor="deepskyblue", alpha=0.7, pad=1),
        )
    ax.set_title(f"Sample {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()